#### 1. Imports, Silver tables, and reporting window

In [1]:
from pyspark.sql import functions as F

silver_customers = spark.table("silver_customers")
silver_depots = spark.table("silver_depots")
silver_rentals = spark.table("silver_rentals")
silver_billing = spark.table("silver_billing")

WINDOW_START_DATE = "2026-06-01"
WINDOW_END_DATE = "2026-06-30"

# Exclusive upper timestamp:
# June 1 00:00:00 through July 1 00:00:00 = exactly 30 days.
WINDOW_START_TS = F.lit(
    "2026-06-01 00:00:00"
).cast("timestamp")

WINDOW_END_EXCLUSIVE_TS = F.lit(
    "2026-07-01 00:00:00"
).cast("timestamp")

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 3, Finished, Available, Finished, False)

In [2]:
assert silver_customers.count() == 600
assert silver_depots.count() == 6
assert silver_rentals.count() == 942
assert silver_billing.count() == 779

print("Silver inputs validated.")

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 4, Finished, Available, Finished, False)

Silver inputs validated.


#### 2. Build gold_dim_customer

##### 2.1 Derive tenure

In [3]:
tenure_years_expression = F.greatest(
    F.lit(0),
    F.floor(
        F.months_between(
            F.to_date(F.lit(WINDOW_END_DATE)),
            F.col("registered_on")
        ) / 12
    ).cast("int")
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 5, Finished, Available, Finished, False)

##### 2.2 Create the customer dimension

In [4]:
gold_dim_customer = (
    silver_customers
    .withColumn(
        "tenure_years",
        tenure_years_expression
    )
    .withColumn(
        "tenure_band",
        F.when(
            F.col("tenure_years") < 1,
            F.lit("Less than 1 year")
        )
        .when(
            F.col("tenure_years") < 3,
            F.lit("1-2 years")
        )
        .when(
            F.col("tenure_years") < 5,
            F.lit("3-4 years")
        )
        .otherwise(
            F.lit("5+ years")
        )
    )
    .select(
        "customer_id",
        "customer_name",
        "registered_on",
        "kyc_verified_on",
        "customer_type",
        "city",
        "tenure_years",
        "tenure_band"
    )
)

print(
    "gold_dim_customer rows:",
    gold_dim_customer.count()
)

display(
    gold_dim_customer
    .groupBy("tenure_band")
    .count()
    .orderBy("tenure_band")
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 6, Finished, Available, Finished, False)

gold_dim_customer rows: 600


SynapseWidget(Synapse.DataFrame, 0c645283-bc8c-405f-9018-5f8e1fc45e73)

In [5]:
assert gold_dim_customer.count() == 600

assert (
    gold_dim_customer
    .filter(F.col("tenure_years").isNull())
    .count()
) == 0

assert (
    gold_dim_customer
    .filter(F.col("tenure_band").isNull())
    .count()
) == 0

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 7, Finished, Available, Finished, False)

#### 3. Build gold_dim_depot

In [6]:
gold_dim_depot = (
    silver_depots
    .select(
        "depot_code",
        "depot_name",
        "zone",
        "fleet_size"
    )
)

display(gold_dim_depot)

assert gold_dim_depot.count() == 6

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 87c63815-f57e-4c1d-b8e1-66ec2df914ee)

#### 4. Generate gold_dim_date

In [7]:
gold_dim_date = (
    spark.range(0, 30)
    .select(
        F.date_add(
            F.to_date(F.lit(WINDOW_START_DATE)),
            F.col("id").cast("int")
        ).alias("calendar_date")
    )
    .withColumn(
        "date_key",
        F.date_format(
            F.col("calendar_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "year",
        F.year("calendar_date")
    )
    .withColumn(
        "quarter",
        F.quarter("calendar_date")
    )
    .withColumn(
        "month_number",
        F.month("calendar_date")
    )
    .withColumn(
        "month_name",
        F.date_format("calendar_date", "MMMM")
    )
    .withColumn(
        "day_of_month",
        F.dayofmonth("calendar_date")
    )
    .withColumn(
        "day_name",
        F.date_format("calendar_date", "EEEE")
    )
    .withColumn(
        "week_of_year",
        F.weekofyear("calendar_date")
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("calendar_date")
        .isin(1, 7)
        .cast("int")
    )
    .select(
        "date_key",
        "calendar_date",
        "year",
        "quarter",
        "month_number",
        "month_name",
        "day_of_month",
        "day_name",
        "week_of_year",
        "is_weekend"
    )
    .orderBy("calendar_date")
)

display(gold_dim_date)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 765435f1-1301-49b0-b7e6-e8be1e7f9c82)

In [11]:
date_summary = (
    gold_dim_date
    .agg(
        F.count("*").alias("row_count"),
        F.min("calendar_date").alias("first_date"),
        F.max("calendar_date").alias("last_date")
    )
)

display(date_summary)

assert gold_dim_date.count() == 30

assert (
    gold_dim_date
    .filter(
        F.col("calendar_date")
        == F.to_date(F.lit(WINDOW_START_DATE))
    )
    .count()
) == 1

assert (
    gold_dim_date
    .filter(
        F.col("calendar_date")
        == F.to_date(F.lit(WINDOW_END_DATE))
    )
    .count()
) == 1

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 59821c52-753b-4eb1-b26a-38fafee426a8)

#### 5. Build gold_fact_rental

The grain remains: One row per rental

#### 5.1 Derive rental duration

In [12]:
rental_duration_expression = (
    F.when(
        F.col("checkin_ts").isNull(),
        F.lit(None)
    )
    .otherwise(
        F.round(
            (
                F.col("checkin_ts").cast("long")
                - F.col("checkout_ts").cast("long")
            ) / 86400.0,
            2
        )
    )
    .cast("decimal(10,2)")
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 14, Finished, Available, Finished, False)

##### 5.2 Clip rented time to the reporting window

In [13]:
effective_window_start = F.greatest(
    F.col("checkout_ts"),
    WINDOW_START_TS
)

effective_window_end = F.least(
    F.coalesce(
        F.col("checkin_ts"),
        WINDOW_END_EXCLUSIVE_TS
    ),
    WINDOW_END_EXCLUSIVE_TS
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 15, Finished, Available, Finished, False)

In [14]:
asset_seconds_in_window = F.greatest(
    F.lit(0),
    effective_window_end.cast("long")
    - effective_window_start.cast("long")
)

asset_days_expression = (
    F.round(
        asset_seconds_in_window / 86400.0,
        2
    )
    .cast("decimal(10,2)")
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 16, Finished, Available, Finished, False)

##### 5.3 Create the rental fact

In [15]:
gold_fact_rental = (
    silver_rentals
    .withColumn(
        "checkout_date_key",
        F.date_format(
            F.to_date("checkout_ts"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "checkin_date_key",
        F.when(
            F.col("checkin_ts").isNotNull(),
            F.date_format(
                F.to_date("checkin_ts"),
                "yyyyMMdd"
            ).cast("int")
        )
    )
    .withColumn(
        "asset_type",
        F.regexp_extract(
            F.col("asset_id"),
            r"^([A-Za-z]+)",
            1
        )
    )
    .withColumn(
        "rental_duration_days",
        rental_duration_expression
    )
    .withColumn(
        "is_returned",
        F.col("checkin_ts")
        .isNotNull()
        .cast("int")
    )
    .withColumn(
        "asset_days_in_window",
        asset_days_expression
    )
    .select(
        "rental_id",
        "customer_id",
        "depot_code",
        "asset_id",
        "asset_type",
        "checkout_date_key",
        "checkin_date_key",
        "checkout_ts",
        "checkin_ts",
        "rental_type",
        "rental_duration_days",
        "is_returned",
        "asset_days_in_window"
    )
)

print(
    "gold_fact_rental rows:",
    gold_fact_rental.count()
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 17, Finished, Available, Finished, False)

gold_fact_rental rows: 942


##### 5.4 Validate the honest rental measures

In [16]:
still_out_count = (
    gold_fact_rental
    .filter(F.col("is_returned") == 0)
    .count()
)

null_duration_count = (
    gold_fact_rental
    .filter(
        F.col("rental_duration_days").isNull()
    )
    .count()
)

returned_with_null_duration = (
    gold_fact_rental
    .filter(
        (F.col("is_returned") == 1)
        & F.col("rental_duration_days").isNull()
    )
    .count()
)

still_out_with_duration = (
    gold_fact_rental
    .filter(
        (F.col("is_returned") == 0)
        & F.col("rental_duration_days").isNotNull()
    )
    .count()
)

asset_days_summary = (
    gold_fact_rental
    .agg(
        F.min("asset_days_in_window").alias(
            "minimum_asset_days"
        ),
        F.max("asset_days_in_window").alias(
            "maximum_asset_days"
        )
    )
)

print(f"Fact rows:                     {gold_fact_rental.count()}")
print(f"Still-out rentals:             {still_out_count}")
print(f"Null rental durations:         {null_duration_count}")
print(f"Returned with null duration:   {returned_with_null_duration}")
print(f"Still-out with duration:       {still_out_with_duration}")

display(asset_days_summary)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 18, Finished, Available, Finished, False)

Fact rows:                     942
Still-out rentals:             175
Null rental durations:         175
Returned with null duration:   0
Still-out with duration:       0


SynapseWidget(Synapse.DataFrame, afb2755c-afa8-44a1-948a-c67b609a92ad)

In [17]:
assert gold_fact_rental.count() == 942

assert still_out_count == 175

assert null_duration_count == 175

assert returned_with_null_duration == 0

assert still_out_with_duration == 0

assert (
    gold_fact_rental
    .filter(
        F.col("asset_days_in_window") > 30
    )
    .count()
) == 0

assert (
    gold_fact_rental
    .filter(
        F.col("asset_days_in_window") < 0
    )
    .count()
) == 0

print("Rental fact validation passed.")

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 19, Finished, Available, Finished, False)

Rental fact validation passed.


###### Inspect the still-out rentals:

In [18]:
display(
    gold_fact_rental
    .filter(F.col("is_returned") == 0)
    .select(
        "rental_id",
        "asset_id",
        "checkout_ts",
        "checkin_ts",
        "rental_duration_days",
        "is_returned",
        "asset_days_in_window"
    )
    .orderBy("checkout_ts")
    .limit(20)
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 34eb3abe-2df5-4fd4-896a-e638a6f95a34)

#### 6. Build gold_fact_billing

In [19]:
gold_fact_billing = (
    silver_billing
    .withColumn(
        "bill_date_key",
        F.date_format(
            F.col("bill_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "bill_id",
        "rental_id",
        "customer_id",
        "depot_code",
        "bill_date_key",
        "bill_date",
        "amount_inr",
        "payer_type"
    )
)

print(
    "gold_fact_billing rows:",
    gold_fact_billing.count()
)

assert gold_fact_billing.count() == 779

assert (
    gold_fact_billing
    .filter(F.col("amount_inr").isNull())
    .count()
) == 0

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 21, Finished, Available, Finished, False)

gold_fact_billing rows: 779


#### 7. Validate the conformed keys

##### Customer keys in rental fact

In [20]:
missing_rental_customers = (
    gold_fact_rental.alias("f")
    .join(
        gold_dim_customer.alias("c"),
        F.col("f.customer_id")
        == F.col("c.customer_id"),
        "left_anti"
    )
    .count()
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 22, Finished, Available, Finished, False)

##### Depot keys in rental fact

In [21]:
missing_rental_depots = (
    gold_fact_rental.alias("f")
    .join(
        gold_dim_depot.alias("d"),
        F.col("f.depot_code")
        == F.col("d.depot_code"),
        "left_anti"
    )
    .count()
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 23, Finished, Available, Finished, False)

##### Keys in billing fact

In [22]:
missing_billing_customers = (
    gold_fact_billing.alias("f")
    .join(
        gold_dim_customer.alias("c"),
        F.col("f.customer_id")
        == F.col("c.customer_id"),
        "left_anti"
    )
    .count()
)

missing_billing_depots = (
    gold_fact_billing.alias("f")
    .join(
        gold_dim_depot.alias("d"),
        F.col("f.depot_code")
        == F.col("d.depot_code"),
        "left_anti"
    )
    .count()
)

print(f"Missing rental customers:  {missing_rental_customers}")
print(f"Missing rental depots:     {missing_rental_depots}")
print(f"Missing billing customers: {missing_billing_customers}")
print(f"Missing billing depots:    {missing_billing_depots}")

assert missing_rental_customers == 0
assert missing_rental_depots == 0
assert missing_billing_customers == 0
assert missing_billing_depots == 0

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 24, Finished, Available, Finished, False)

Missing rental customers:  0
Missing rental depots:     0
Missing billing customers: 0
Missing billing depots:    0


#### 8. Write the five Gold tables

In [23]:
def write_gold_table(dataframe, table_name):
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Created: {table_name}")

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 25, Finished, Available, Finished, False)

In [24]:
gold_tables = {
    "gold_dim_customer": gold_dim_customer,
    "gold_dim_depot": gold_dim_depot,
    "gold_dim_date": gold_dim_date,
    "gold_fact_rental": gold_fact_rental,
    "gold_fact_billing": gold_fact_billing
}

for table_name, dataframe in gold_tables.items():
    write_gold_table(
        dataframe,
        table_name
    )

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 26, Finished, Available, Finished, False)

Created: gold_dim_customer
Created: gold_dim_depot
Created: gold_dim_date
Created: gold_fact_rental
Created: gold_fact_billing


#### 9. Final row-count validation

In [25]:
expected_gold_counts = {
    "gold_dim_customer": 600,
    "gold_dim_depot": 6,
    "gold_dim_date": 30,
    "gold_fact_rental": 942,
    "gold_fact_billing": 779
}

for table_name, expected_count in expected_gold_counts.items():
    actual_count = spark.table(table_name).count()

    print(
        f"{table_name:<26} "
        f"expected={expected_count:<4} "
        f"actual={actual_count:<4}"
    )

    assert actual_count == expected_count, (
        f"{table_name}: expected {expected_count}, "
        f"found {actual_count}"
    )

print("\nAll Gold row counts passed.")

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 27, Finished, Available, Finished, False)

gold_dim_customer          expected=600  actual=600 
gold_dim_depot             expected=6    actual=6   
gold_dim_date              expected=30   actual=30  
gold_fact_rental           expected=942  actual=942 
gold_fact_billing          expected=779  actual=779 

All Gold row counts passed.


#### 10. Run OPTIMIZE and ZORDER

In [26]:
display(
    spark.sql("""
        OPTIMIZE gold_fact_rental
        ZORDER BY (depot_code, checkout_date_key)
    """)
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f726b446-a3ae-41d5-9df4-b383332f7660)

In [27]:
display(
    spark.sql("""
        OPTIMIZE gold_fact_billing
        ZORDER BY (depot_code, bill_date_key)
    """)
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5b7e7cdb-6585-4aa0-a043-7d1fa4c7d061)

#### 11. Show DESCRIBE HISTORY

##### Rental fact history

In [28]:
rental_history = spark.sql("""
    DESCRIBE HISTORY gold_fact_rental
""")

display(
    rental_history
    .select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    )
    .orderBy(F.desc("version"))
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d87f93fc-3d0e-4fb0-bf71-e0c851ffeef8)

##### Billing fact history

In [29]:
billing_history = spark.sql("""
    DESCRIBE HISTORY gold_fact_billing
""")

display(
    billing_history
    .select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    )
    .orderBy(F.desc("version"))
)

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 063348bd-2bdb-4c19-b340-b532ba5bb59d)

##### Validate that the history includes the operation:

In [30]:
rental_optimize_operations = (
    rental_history
    .filter(
        F.upper(F.col("operation")) == "OPTIMIZE"
    )
    .count()
)

billing_optimize_operations = (
    billing_history
    .filter(
        F.upper(F.col("operation")) == "OPTIMIZE"
    )
    .count()
)

print(
    "Rental OPTIMIZE operations:",
    rental_optimize_operations
)

print(
    "Billing OPTIMIZE operations:",
    billing_optimize_operations
)

assert rental_optimize_operations >= 1
assert billing_optimize_operations >= 1

print("OPTIMIZE is present in both Delta histories.")

StatementMeta(, 32e3cc82-78de-4095-9454-db84f8040d58, 32, Finished, Available, Finished, False)

Rental OPTIMIZE operations: 1
Billing OPTIMIZE operations: 1
OPTIMIZE is present in both Delta histories.


Partitioning: I did not partition the Gold facts because they contain only 942 and 779 rows across a thirty-day window; partitioning such small tables would add metadata and small-file overhead without a practical performance benefit.
Honest duration: rental_duration_days remains null while a machine is still out because its final duration is not yet known; using zero would falsely represent an immediate return and corrupt duration averages.